In [1]:
# from sklearn.model_selection import GridSearchCV
# from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, roc_curve, auc
# from sklearn import svm
# from sklearn.ensemble import RandomForestClassifier
# from src.processing import Processing
import config
# from sklearn.feature_selection import SelectKBest, chi2
# import matplotlib.pyplot as plt
# import seaborn as sns
# import joblib
# import pandas as pd
# import numpy as np
# from imblearn.over_sampling import SMOTE
# from imblearn.under_sampling import RandomUnderSampler
import warnings
import random
# from src.datasplitting import DataSplitting
warnings.filterwarnings('ignore')
# from sklearn.model_selection import GridSearchCV, PredefinedSplit, TimeSeriesSplit
# # from sklearn.metrics import (
# #     accuracy_score, f1_score, precision_score, recall_score,
# #     confusion_matrix, roc_auc_score, precision_recall_curve, auc, classification_report
# )
# from src.processing import Processing
# from sklearn.metrics import roc_curve, precision_recall_curve, average_precision_score
# from sklearn.decomposition import PCA
# from sklearn.neighbors import KNeighborsClassifier
# from imblearn.under_sampling import EditedNearestNeighbours
# from imblearn.pipeline import make_pipeline, Pipeline
# from sklearn.preprocessing import StandardScaler
import numpy as np
# import pandas as pd
# import matplotlib.pyplot as plt
import os
# from imblearn.over_sampling import SMOTE
# from collections import Counter
import tensorflow as tf
from tensorflow.keras.applications import VGG16
from tensorflow.keras import layers, models
import os
import gc
from tfrecordmultimodalhandler import TFRecordMultimodalHandler
from tensorflow.random import set_seed
# from tfrecordhandler import TFRecordDataHandler
# from src.datasplitting_for_images import DataSplittingForImages

In [2]:
SEED = 42

In [3]:
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)

In [4]:
import matplotlib.pyplot as plt
import numpy as np

def visualize_data(image_batch, days_of_study_batch, labels_batch, num_samples=3):
    plt.figure(figsize=(10, num_samples * 3))

    for i in range(num_samples):
        # Primer canal: la imagen en escala de grises
        plt.subplot(num_samples, 2, 2 * i + 1)
        plt.imshow(np.squeeze(image_batch[i, :, :, 0]), cmap='gray')  # Primer canal
        plt.title(f"Image Day {days_of_study_batch[i]} - Label {labels_batch[i]}")
        plt.axis('off')

        # Segundo canal: la máscara
        plt.subplot(num_samples, 2, 2 * i + 2)
        plt.imshow(np.squeeze(image_batch[i, :, :, 1]), cmap='gray')  # Segundo canal
        plt.title(f"Mask ")
        plt.axis('off')

    plt.show()

In [5]:
# Definir hiperparámetros
INPUT_DIR = 'input/deep_learning'
batch_size = 1
n_epochs = 10
checkpoint = './best_model_vgg.h5'
shuffle = True
verbose = 1

In [6]:
# def _bytes_feature(value):
#     """Returns a bytes_list from a string / byte."""
#     return tf.train.Feature(bytes_list=tf.train.BytesList(value=[value]))

# def _int64_feature(value):
#     """Returns an int64_list from a bool / enum / int / uint."""
#     return tf.train.Feature(int64_list=tf.train.Int64List(value=[value]))

# def serialize_example(image_path, mask_path):
#     img = tf.io.decode_image(tf.io.read_file(image_path), channels=3)
#     img_bytes = img.numpy().tobytes()

#     mask = tf.io.decode_image(tf.io.read_file(mask_path), channels=1)
#     mask_bytes = mask.numpy().tobytes()

#     # Get Study, Mice group and Day of study from the path
#     path = os.path.normpath(mask_path)
    
#     # Divide el path en partes
#     path_parts = path.split(os.sep)

#     # Extraer las partes relevantes
#     # 0 for Cured Mice, 1 for Relapsing Mice, 2 for Control Mice
#     if path_parts[2] == 'Cured mice':
#         group_name = 1
#     elif path_parts[2] == 'IMS-TMS-TREATED-RELAPSING':
#         group_name = 0
#     elif path_parts[2] == 'Control':
#         group_name = 0
#     else:
#         print(path_parts)
#         raise ValueError('Invalid group')
#     mouse_id = path_parts[3]       # 'C1281'
#     day_of_study = path_parts[4]   # 'day10'

#     # Get only the integers in ids
#     mouse_id = mouse_id[1:]
#     day_of_study = day_of_study[3:]

#     feature = {
#         'image': _bytes_feature(img_bytes),
#         'mask': _bytes_feature(mask_bytes),
#         'height': _int64_feature(img.shape[0]),
#         'width': _int64_feature(img.shape[1]),
#         'mask_height': _int64_feature(mask.shape[0]),
#         'mask_width': _int64_feature(mask.shape[1]),
#         'group_name': _int64_feature(int(group_name)),
#         'mouse_id': _int64_feature(int(mouse_id)),
#         'day_of_study': _int64_feature(int(day_of_study))
#     }
    
#     return tf.train.Example(features=tf.train.Features(feature=feature)).SerializeToString()

# def create_tfrecord(file_paths, output_file):
#     with tf.io.TFRecordWriter(output_file) as writer:
#         for img_path, mask_path in file_paths:
#             if not os.path.exists(img_path) or not os.path.exists(mask_path):
#                 print(f'Image or mask not found: {img_path}, {mask_path}')
#                 continue
#             writer.write(serialize_example(img_path, mask_path))

# def split_dataset_by_m_id(root_dir, train_ratio=0.8):
#     mouse_to_data = {}
    
#     # Recorrer todas las imágenes y máscaras
#     for data_group in os.listdir(root_dir):
#         data_group_path = os.path.join(root_dir, data_group)
#         if not os.path.isdir(data_group_path):
#             continue
#         for mice_group in os.listdir(data_group_path):
#             mice_group_path = os.path.join(data_group_path, mice_group)
#             if not os.path.isdir(mice_group_path):
#                 continue  # Saltar si no es un directorio
#             for dayofstudy in os.listdir(mice_group_path):
#                 dayofstudy_path = os.path.join(mice_group_path, dayofstudy)
#                 if not os.path.isdir(dayofstudy_path):
#                     continue  # Saltar si no es un directorio
#                 mri_imgs = os.path.join(dayofstudy_path, 'MRI images')
#                 if not os.path.exists(mri_imgs):
#                     print(f"No se encontró la carpeta {mri_imgs}")
#                     continue
#                 # Número de imágenes
#                 n_imgs = len(os.listdir(mri_imgs))
#                 for i in range(n_imgs):
#                     img_path = os.path.join(mri_imgs, f'image_s{i + 1}.jpg')
#                     mask_path = os.path.join(dayofstudy_path, f'Mask s{i + 1}.jpg')
                    
#                     # Get Study, Mice group and Day of study from the path
#                     path = os.path.normpath(mask_path)
                    
#                     # Divide el path en partes
#                     path_parts = path.split(os.sep)

#                     # Extraer las partes relevantes
#                     # 0 for Relapsing Mice, 1 for Cured Mice, 2 for Control Mice
#                     if path_parts[2] == 'Cured mice':
#                         group_name = 1
#                     elif path_parts[2] == 'IMS-TMS-TREATED-RELAPSING':
#                         group_name = 0
#                     elif path_parts[2] == 'Control':
#                         group_name = 0
#                     else:
#                         raise ValueError('Invalid group')
                    
#                     mouse_id = path_parts[3]       # 'C1281'
#                     day_of_study = path_parts[4]   # 'day10'

#                     # Get only the integers in ids
#                     mouse_id = mouse_id[1:]  # Remove 'C'
#                     day_of_study = day_of_study[3:]  # Remove 'day'
                    
#                     # Append data to mouse ID group
#                     if mouse_id not in mouse_to_data:
#                         mouse_to_data[mouse_id] = []
#                     mouse_to_data[mouse_id].append((img_path, mask_path, group_name, mouse_id, day_of_study))

#     print(mouse_to_data.head(2))

#     # Split mouse IDs into train and test sets
#     all_mice = list(mouse_to_data.keys())
#     random.shuffle(all_mice)  # Shuffle IDs for randomness

#     # Identify cured mice
#     cured_mice = [mouse_id for mouse_id in all_mice if mouse_to_data[mouse_id][0][2] == 1]

#     # Ensure at least one cured mouse in both train and test sets
#     train_size = int(len(all_mice) * train_ratio)

#     # Randomly shuffle cured mice to ensure diversity
#     random.shuffle(cured_mice)

#     # Ensure that at least one cured mouse is in the train and test sets
#     train_mice = set(all_mice[:train_size])
#     test_mice = set(all_mice[train_size:])

#     # If the train set doesn't have a cured mouse, move one from test
#     if not any(mouse_id in train_mice for mouse_id in cured_mice):
#         train_mice.add(cured_mice[0])
#         test_mice.remove(cured_mice[0])

#     # If the test set doesn't have a cured mouse, move one from train
#     if not any(mouse_id in test_mice for mouse_id in cured_mice):
#         test_mice.add(cured_mice[0])
#         train_mice.remove(cured_mice[0])

#     # Check that both train and test sets have a cured mouse
#     print(f'Train mice: {train_mice}')
#     print(f'Test mice: {test_mice}')

#     # Assign data to train and test sets
#     train_data = [data for mouse_id in train_mice for data in mouse_to_data[mouse_id]]
#     test_data = [data for mouse_id in test_mice for data in mouse_to_data[mouse_id]]

#     train_paths = [(img, mask) for img, mask, *_ in train_data]
#     test_paths = [(img, mask) for img, mask, *_ in test_data]

#     print(f"Total data: {len(train_data) + len(test_data)}, Train: {len(train_data)}, Test: {len(test_data)}")
    
#     return train_paths, test_paths


In [7]:
# train_data, test_data = split_dataset_by_m_id(config.DATASET_DIR, train_ratio=0.7)
# train_tfrefcord = os.path.join(INPUT_DIR, 'train.tfrecord')
# test_tfrefcord = os.path.join(INPUT_DIR, 'test.tfrecord')

# create_tfrecord(train_data, train_tfrefcord)
# create_tfrecord(test_data, test_tfrefcord)

In [8]:
train_tfrefcord = os.path.join(INPUT_DIR, 'train_new.tfrecord')
# test_tfrefcord = os.path.join(INPUT_DIR, 'test.tfrecord')

In [9]:
train_ds_handler = TFRecordMultimodalHandler(train_tfrefcord, batch_size=batch_size, shuffle=False, augment=False)
train_ds = train_ds_handler.dataset
print(f"Number of train samples: {train_ds_handler.length}")

# test_ds_handler = TFRecordMultimodalHandler(test_tfrefcord, batch_size=batch_size, shuffle=False, augment=False)
# test_ds = test_ds_handler.dataset
# print(f"Number of test samples: {test_ds_handler.length}")

# Number of batches
n_batches = train_ds_handler.length // batch_size
print(f"Number of batches: {n_batches}")

dataset_img, labels = next(iter(train_ds))
print(dataset_img["image_input"])  

# Check if images are normalized
print(f"Images min: {dataset_img["image_input"].numpy().min()}, max: {dataset_img["image_input"].numpy().max()}")

# visualize_data(images_tf, num_samples=4)
# visualize_data(dataset_img["image_input"], dataset_img["day_input"], labels, num_samples=15)


Number of train samples: 298
Number of batches: 298
tf.Tensor(
[[[[[0.00784314 1.         1.        ]
    [0.00784314 1.         1.        ]
    [0.00392157 1.         1.        ]
    ...
    [0.01960784 1.         1.        ]
    [0.01568628 1.         1.        ]
    [0.01176471 1.         1.        ]]

   [[0.00784314 1.         1.        ]
    [0.00784314 1.         1.        ]
    [0.00392157 1.         1.        ]
    ...
    [0.01176471 1.         1.        ]
    [0.00784314 1.         1.        ]
    [0.00392157 1.         1.        ]]

   [[0.01176471 1.         1.        ]
    [0.00784314 1.         1.        ]
    [0.00392157 1.         1.        ]
    ...
    [0.01176471 1.         1.        ]
    [0.01176471 1.         1.        ]
    [0.01176471 1.         1.        ]]

   ...

   [[0.01176471 1.         1.        ]
    [0.01176471 1.         1.        ]
    [0.01176471 1.         1.        ]
    ...
    [0.01568628 1.         1.        ]
    [0.01568628 1.         1.    

In [10]:
print(dataset_img["image_input"].shape[1])

56


In [11]:
# import numpy as np
# import tensorflow as tf

# mice_in_train = {}

# for ds_batch, labels_batch in train_ds:  # Itera sobre los batches del dataset
#     for i in range(len(labels_batch)):  # Itera sobre cada instancia dentro del batch
#         image_shape = ds_batch["image_input"][i].shape  # Obtener la forma de la imagen
#         image_timesteps = ds_batch["image_input"][i].shape[0]  # Obtener la cantidad de timesteps
#         day = ds_batch["day_input"][i].numpy()  # Día de estudio
#         mouse_id = ds_batch["mouse_id"][i].numpy()  # ID del ratón
#         label = labels_batch[i].numpy()  # Etiqueta de clase
#         if mouse_id not in mice_in_train:
#             mice_in_train[mouse_id] = [[], []]
#         mice_in_train[mouse_id][0].append(day)
#         if not mice_in_train[mouse_id][1]:
#             mice_in_train[mouse_id][1] = image_timesteps

# print(mice_in_train)

# for mouse_id, values in mice_in_train.items():
#     # print(f"Mouse ID: {mouse_id}, Days: {set(values[0])}, Length: {len(values[0])} - Timesteps: {values[1]}")
#     print(f"Mouse ID: {mouse_id}, Values: {values}")
    # if len((values[0])) != values[1]:
    #     print(f"Mouse ID: {mouse_id}, Days: {set(values[0])}, Length: {len(set(values[0]))} - Timesteps: {values[1]}")
    #     raise ValueError('Days and timesteps do not match')

# for ds_batch, labels_batch in train_ds:  # Add new dimension in position 1 to the images with the length of the sequence of days
#     batch_size = ds_batch["image_input"].shape[0]
#     for i in range(batch_size):
#         mouse_id = ds_batch["mouse_id"][i].numpy()
#         timesteps = len(mice_in_train[mouse_id])
#         print(ds_batch["image_input"].shape)
#         ds_batch["image_input"] = np.tile(tf.expand_dims(ds_batch["image_input"], axis=1), (1, timesteps, 1, 1, 1))
#         print(ds_batch["image_input"].shape)
    


In [12]:
# # import numpy as np
# # import tensorflow as tf

# mice_in_train = {}
# map_timesteps = {}
# for ds_batch, labels_batch in test_ds:  # Itera sobre los batches del dataset
#     for i in range(len(labels_batch)):  # Itera sobre cada instancia dentro del batch
#         image_shape = ds_batch["image_input"][i].shape  # Obtener la forma de la imagen
#         day = ds_batch["day_input"][i].numpy()  # Día de estudio
#         mouse_id = ds_batch["mouse_id"][i].numpy()  # ID del ratón
#         label = labels_batch[i].numpy()  # Etiqueta de clase
#         if mouse_id not in mice_in_train:
#             mice_in_train[mouse_id] = []
#         mice_in_train[mouse_id].append(day)

# for mouse_id, days in mice_in_train.items():
#     print(f"Mouse ID: {mouse_id}, Days: {set(days)}, Length: {len(days)}")

# for ds_batch, labels_batch in test_ds:  # Add new dimension in position 1 to the images with the length of the sequence of days
#     batch_size = ds_batch["image_input"].shape[0]
#     for i in range(batch_size):
#         mouse_id = ds_batch["mouse_id"][i].numpy()
#         timesteps = len(mice_in_train[mouse_id])
#         ds_batch["image_input"] = np.tile(tf.expand_dims(ds_batch["image_input"], axis=1), (1, timesteps, 1, 1, 1))


In [13]:
# import numpy as np
# import tensorflow as tf

# mice_in_train = {}

# # Procesar train_ds
# updated_train_ds = []
# for ds_batch, labels_batch in train_ds:
#     for i in range(len(labels_batch)):
#         day = ds_batch["day_input"][i].numpy()
#         mouse_id = ds_batch["mouse_id"][i].numpy()
#         if mouse_id not in mice_in_train:
#             mice_in_train[mouse_id] = []
#         mice_in_train[mouse_id].append(day)

# print(type(train_ds))
# for ds_batch, labels_batch in train_ds:
#     batch_size = ds_batch["image_input"].shape[0]
#     updated_images = []
#     for i in range(batch_size):
#         mouse_id = ds_batch["mouse_id"][i].numpy()
#         timesteps = len(mice_in_train[mouse_id])
#         expanded_images = np.tile(tf.expand_dims(ds_batch["image_input"], axis=1), (1, timesteps, 1, 1, 1))
#         updated_images.append(expanded_images)
#     ds_batch["image_input"] = np.concatenate(updated_images, axis=0)
#     updated_train_ds.append((ds_batch, labels_batch))

# train_ds = updated_train_ds  # Reasignar el dataset actualizado

# # Procesar test_ds
# updated_test_ds = []
# mice_in_test = {}
# for ds_batch, labels_batch in test_ds:
#     for i in range(len(labels_batch)):
#         day = ds_batch["day_input"][i].numpy()
#         mouse_id = ds_batch["mouse_id"][i].numpy()
#         if mouse_id not in mice_in_test:
#             mice_in_test[mouse_id] = []
#         mice_in_test[mouse_id].append(day)

# for ds_batch, labels_batch in test_ds:
#     batch_size = ds_batch["image_input"].shape[0]
#     updated_images = []
#     for i in range(batch_size):
#         mouse_id = ds_batch["mouse_id"][i].numpy()
#         timesteps = len(mice_in_test[mouse_id])
#         expanded_images = np.tile(tf.expand_dims(ds_batch["image_input"], axis=1), (1, timesteps, 1, 1, 1))
#         updated_images.append(expanded_images)
#     ds_batch["image_input"] = np.concatenate(updated_images, axis=0)
#     updated_test_ds.append((ds_batch, labels_batch))

# test_ds = updated_test_ds  # Reasignar el dataset actualizado


In [14]:

tf.random.set_seed(SEED)

# Oversampling with data augmentation
def augment_data(dataset_img, label):
    image = tf.image.random_flip_left_right(dataset_img["image_input"])
    image = tf.image.random_brightness(dataset_img["image_input"], max_delta=0.2)
    return {"image_input": image, "day_input": dataset_img["day_input"]}, label

def count_instances(dataset):
    total_count = 0
    for _, labels in dataset:
        total_count += tf.size(labels)  # Count the number of labels (which equals the number of images in each batch)
    return total_count.numpy()  # Convert TensorFlow tensor to a regular integer

def oversample_minority_class(dataset, minority_class=1, factor=2):
    # Separate the minority and majority classes
    minority_ds = dataset.filter(lambda _, lbl: tf.reduce_any(lbl == minority_class))
    majority_ds = dataset.filter(lambda _, lbl: tf.reduce_any(lbl != minority_class))

    # Count instances before augmentation
    minority_count = count_instances(minority_ds)
    majority_count = count_instances(majority_ds)

    print(f"Before augmentation, Minority class size: {minority_count}, Majority class size: {majority_count}")
    
    # Augment the minority class data
    augmented_minority_ds = minority_ds.map(augment_data)
    for _ in range(factor - 1):
        augmented_minority_ds = augmented_minority_ds.concatenate(minority_ds.map(augment_data))

    # Recombine and shuffle
    balanced_ds = majority_ds.concatenate(augmented_minority_ds)
    
    # After oversampling, count again
    new_minority_count = count_instances(augmented_minority_ds)
    new_majority_count = count_instances(majority_ds)
    
    print(f"After augmentation, Minority class size: {new_minority_count}, Majority class size: {new_majority_count}")
    
    return balanced_ds.shuffle(buffer_size=1000, seed=SEED)

# Apply oversampling to the dataset
# train_ds = oversample_minority_class(train_ds, minority_class=1, factor=3)

In [15]:
# dataset_img, labels = train_ds.take(15).__iter__().__next__()

# Print shape of images and masks
print(f"Images shape: {dataset_img["image_input"].shape}")
print(f"Days of study shape: {dataset_img["day_input"].shape}")
print(f"Days of study shape: {dataset_img["mouse_id"].shape}")
print(f"Labels shape: {labels.shape}")

Images shape: (1, 56, 256, 256, 3)
Days of study shape: (1, 56)
Days of study shape: (1,)
Labels shape: (1,)


In [16]:
for ds, label in train_ds:
    print(ds["image_input"].shape)
    print(ds["mouse_id"])
    print(label)

(1, 56, 256, 256, 3)
tf.Tensor([1263], shape=(1,), dtype=int64)
tf.Tensor([0], shape=(1,), dtype=int32)
(1, 43, 256, 256, 3)
tf.Tensor([1264], shape=(1,), dtype=int64)
tf.Tensor([0], shape=(1,), dtype=int32)
(1, 43, 256, 256, 3)
tf.Tensor([1270], shape=(1,), dtype=int64)
tf.Tensor([0], shape=(1,), dtype=int32)
(1, 21, 256, 256, 3)
tf.Tensor([1276], shape=(1,), dtype=int64)
tf.Tensor([1], shape=(1,), dtype=int32)
(1, 12, 256, 256, 3)
tf.Tensor([1281], shape=(1,), dtype=int64)
tf.Tensor([1], shape=(1,), dtype=int32)
(1, 8, 256, 256, 3)
tf.Tensor([1284], shape=(1,), dtype=int64)
tf.Tensor([1], shape=(1,), dtype=int32)
(1, 17, 256, 256, 3)
tf.Tensor([1285], shape=(1,), dtype=int64)
tf.Tensor([1], shape=(1,), dtype=int32)
(1, 47, 256, 256, 3)
tf.Tensor([1380], shape=(1,), dtype=int64)
tf.Tensor([0], shape=(1,), dtype=int32)
(1, 8, 256, 256, 3)
tf.Tensor([1382], shape=(1,), dtype=int64)
tf.Tensor([1], shape=(1,), dtype=int32)
(1, 43, 256, 256, 3)
tf.Tensor([1383], shape=(1,), dtype=int64)
tf

In [17]:
# for batch, l in train_ds:
#     print(batch)  # Revisar la estructura del dataset antes de pasarlo al modelo


In [18]:
# for batch in train_ds.take(1):  
#     print(batch)  # Revisar la estructura del dataset antes de pasarlo al modelo 


In [19]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import os
from datetime import datetime
from tensorflow.keras.applications import VGG16
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, TensorBoard
from tensorflow.keras.metrics import AUC
import cv2
import gc

# 📌 Data Augmentation
def augment_image(image):
    """Aplica data augmentation a una imagen."""
    image = tf.image.random_flip_left_right(image)
    image = tf.image.random_brightness(image, max_delta=0.2)
    image = tf.image.random_contrast(image, lower=0.8, upper=1.2)
    return image

# 📌 Definir inputs sin `num_timesteps` fijo
image_input = tf.keras.Input(shape=(None, 256, 256, 3), name="image_input")  # Secuencia variable de imágenes
# day_input = tf.keras.Input(shape=(None, 1), name="day_input")  # Secuencia variable de días

# 📌 Aplicar Masking para ignorar valores rellenados
# masked_input = layers.Masking(mask_value=0.0)(image_input)

# 📌 CNN para extracción de características
base_model = VGG16(weights='imagenet', include_top=False, input_shape=(256, 256, 3))
for layer in base_model.layers:
    layer.trainable = False

# Aplicar la CNN a cada imagen de la secuencia
cnn = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D()
], name="cnn_feature_extractor")

# 📌 TimeDistributed para aplicar CNN a cada imagen de la secuencia
x = layers.TimeDistributed(cnn)(image_input)  # Salida: (batch_size, variable_timesteps, feature_dim)

# 📌 Concatenar la información del día con las features de la CNN
# lstm_input = layers.Concatenate(axis=-1)([x, day_input])  # Salida: (batch_size, variable_timesteps, feature_dim + 1)

# 📌 LSTM para aprendizaje temporal
x = layers.LSTM(64, return_sequences=False)(x)

# Capas densas finales
# x = layers.Dense(64, activation="relu")(x)
# x = layers.Dropout(0.3)(x)
# output = layers.Dense(1, activation="sigmoid")(x)  # Clasificación binaria

# # 📌 Definir el modelo
# model = models.Model(inputs=[image_input], outputs=output)
# model.summary()

import xgboost as xgb
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import train_test_split

from sklearn.model_selection import GroupKFold
import xgboost as xgb
from sklearn.metrics import accuracy_score, roc_auc_score
import numpy as np

# ⚙️ 1. Modelo extractor hasta LSTM
feature_extractor = models.Model(inputs=image_input, outputs=x)  # x = salida LSTM

# ⚙️ 2. Extraer features, labels y grupos (m_id)
def extract_features_labels_groups(dataset):
    
    features_list = []
    labels_list = []
    groups_list = []

    for batch in dataset:
        x_batch, y_batch = batch
        feats = feature_extractor.predict(x_batch["image_input"])
        print(feats.shape)
        features_list.append(feats)
        labels_list.append(y_batch.numpy())
        groups_list.append(x_batch["mouse_id"].numpy())  # <- asegúrate que "m_id" existe como feature

    X = np.concatenate(features_list, axis=0)
    y = np.concatenate(labels_list, axis=0)
    groups = np.concatenate(groups_list, axis=0)
    print(groups)
    return X, y, groups

# ✅ Extraer datos
X, y, groups = extract_features_labels_groups(train_ds)
from sklearn.model_selection import LeaveOneGroupOut

logo = LeaveOneGroupOut()

fold = 1

from sklearn.metrics import confusion_matrix, accuracy_score, roc_auc_score
import xgboost as xgb

accuracies = []
aucs = []

all_y_true = []
all_y_pred = []
all_y_proba = []

well_classified = []
misclassified = []

for train_idx, test_idx in logo.split(X, y, groups=groups):
    print(f"\n🔁 Fold {fold} (Test group: {groups[test_idx][0]})")

    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    xgb_clf = xgb.XGBClassifier(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=4,
        subsample=0.8,
        colsample_bytree=0.8,
        use_label_encoder=False,
        eval_metric='logloss',
        random_state=42
    )

    xgb_clf.fit(X_train, y_train)

    y_pred = xgb_clf.predict(X_test)
    y_proba = xgb_clf.predict_proba(X_test)[:, 1]

    acc = accuracy_score(y_test, y_pred)
    # auc = roc_auc_score(y_test, y_proba)

    accuracies.append(acc)
    # aucs.append(auc)

    for i, (yt, yp) in enumerate(zip(y_test, y_pred)):
        idx = test_idx[i]
        if yt == yp:
            well_classified.append(idx)
            print(f"Well classified: {idx}, y_true: {yt}, y_pred: {yp}")
        else:
            misclassified.append(idx)
            print(f"Misclassified: {idx}, y_true: {yt}, y_pred: {yp}")

    all_y_true.extend(y_test)
    all_y_pred.extend(y_pred)
    all_y_proba.extend(y_proba)

    print(f"✅ Fold {fold} - Accuracy: {acc:.4f},")
    fold += 1

# 📊 Resultados finales
all_y_true = np.array(all_y_true)
all_y_pred = np.array(all_y_pred)

tn, fp, fn, tp = confusion_matrix(all_y_true, all_y_pred).ravel()
accuracy = (tp + tn) / (tp + tn + fp + fn)
sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
auc_final = roc_auc_score(all_y_true, all_y_proba)

print("\n📈 Resultados Finales:")
print(f"✅ Accuracy promedio crossval: {np.mean(accuracies):.4f}")
print(f"✅ AUC promedio crossval: {np.mean(aucs):.4f}")
print(f"✅ Accuracy total: {accuracy:.4f}")
print(f"✅ AUC total: {auc_final:.4f}")
print(f"🔍 Sensitivity (Recall): {sensitivity:.4f}")
print(f"🔍 Specificity: {specificity:.4f}")
print(f"🟢 TP: {tp}, 🟡 FP: {fp}, 🔴 FN: {fn}, 🔵 TN: {tn}")

print(f"\n✅ Bien clasificados: {len(well_classified)}")
print(f"❌ Mal clasificados: {len(misclassified)}")


1/1 ━━━━━━━━━━━━━━━━━━━━ 7s 7s/step
(1, 64)


KeyboardInterrupt: 

In [ ]:
model_save_path = "output/dl/best_model.keras"

model = tf.keras.models.load_model(model_save_path)

ValueError: File not found: filepath=output/dl/best_model.keras. Please ensure the file is an accessible `.keras` zip file.

In [ ]:
for layer in model.layers:
    print(layer.name)

In [ ]:
from src.notebooks.dl.gradcam import GradCAM
import matplotlib.pyplot as plt
import numpy as np
import cv2

for image_batch, labels_batch in test_ds:
    batch_size = image_batch.shape[0]
    fig, axes = plt.subplots(5, batch_size, figsize=(3 * batch_size, 10))

    for i, (image, label) in enumerate(zip(image_batch, labels_batch)):
        image = image.numpy()  # Convertir tensor a NumPy array
        label = int(label.numpy())  # Asegurar que la etiqueta sea un entero

        # Imagen en escala de grises
        axes[0, i].imshow(np.squeeze(image[:, :, 0]), cmap='gray')
        axes[0, i].set_title(f"Image, label: {label}")
        axes[0, i].axis('off')

        # Máscara
        axes[1, i].imshow(np.squeeze(image[:, :, 1]), cmap='gray')
        axes[1, i].set_title("Mask")
        axes[1, i].axis('off')

        # GradCAM Heatmap
        gradcam = GradCAM(model, classIdx=label, layerName='block5_conv3')
        heatmap = gradcam.compute_heatmap(np.expand_dims(image, axis=0))
        axes[2, i].imshow(heatmap, cmap='jet')
        axes[2, i].set_title("GradCAM Heatmap")
        axes[2, i].axis('off')

        # GradCAM Overlay en imagen original
        grayscale_image = (image[:, :, 0] * 255).astype(np.uint8)  # Convertir a escala de grises uint8
        grayscale_image = cv2.cvtColor(grayscale_image, cv2.COLOR_GRAY2RGB)  # Convertir a RGB
        overlay = gradcam.overlay_heatmap(heatmap, grayscale_image)[1]
        axes[3, i].imshow(overlay)
        axes[3, i].set_title("GradCAM Overlay")
        axes[3, i].axis('off')

        # GradCAM Overlay en máscara
        mask_image = (image[:, :, 1] * 255).astype(np.uint8)  # Convertir máscara a uint8
        mask_image = cv2.cvtColor(mask_image, cv2.COLOR_GRAY2RGB)  # Convertir a RGB
        overlay_mask = gradcam.overlay_heatmap(heatmap, mask_image)[1]
        axes[4, i].imshow(overlay_mask)
        axes[4, i].set_title("GradCAM on Mask")
        axes[4, i].axis('off')
    
    plt.tight_layout()
    plt.show()



In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import cv2
from tensorflow.keras import backend as K

# Function to get Grad-CAM heatmap
def get_gradcam_heatmap(model, img_array, class_index, layer_name="block5_conv3"):
    # Get the gradient of the class with respect to the last convolutional layer
    with tf.GradientTape() as tape:
        # The input image is the batch with shape (1, 256, 256, 3)
        last_conv_layer = model.get_layer(layer_name)
        tape.watch(last_conv_layer.output)
        preds = model(img_array)
        class_output = preds[:, class_index]
    
    # Get gradients of the predicted class with respect to the last conv layer
    grads = tape.gradient(class_output, last_conv_layer.output)
    
    # Pool the gradients across all the axes (height, width)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    
    # Multiply each channel in the feature map by the mean gradient
    last_conv_layer_output = last_conv_layer.output[0]
    for i in range(last_conv_layer_output.shape[-1]):
        last_conv_layer_output[:, :, i] *= pooled_grads[i]
    
    # Create the heatmap by averaging over all channels
    heatmap = tf.reduce_mean(last_conv_layer_output, axis=-1)
    
    # Normalize the heatmap
    heatmap = np.maximum(heatmap, 0)
    heatmap /= np.max(heatmap)
    
    return heatmap

# Function to display Grad-CAM results
def display_gradcam(img, heatmap, alpha=0.4):
    # Resize heatmap to match the image dimensions
    heatmap = cv2.resize(heatmap, (img.shape[1], img.shape[0]))
    
    # Convert heatmap to RGB
    heatmap = np.uint8(255 * heatmap)
    heatmap = cv2.applyColorMap(heatmap, cv2.COLORMAP_JET)
    
    # Overlay heatmap on the image
    superimposed_img = cv2.addWeighted(img, 1 - alpha, heatmap, alpha, 0)
    
    return superimposed_img

# Function to generate Grad-CAM for a sample from the test set
def gradcam_visualization(model, test_ds, n_images=3):
    # Pick some images from the test dataset
    test_images, test_labels = next(iter(test_ds))
    
    # Get class indices (assumes binary classification here)
    class_indices = np.argmax(test_labels, axis=1)
    
    # Plot results for the first few images
    for i in range(n_images):
        img = test_images[i].numpy().astype("uint8")  # Convert to numpy for display
        true_label = class_indices[i]
        
        # Get Grad-CAM heatmap
        heatmap = get_gradcam_heatmap(model, test_images[i:i+1], true_label)
        
        # Overlay Grad-CAM heatmap on the image
        superimposed_img = display_gradcam(img, heatmap)
        
        # Plot original image and Grad-CAM heatmap
        plt.figure(figsize=(12, 6))
        
        # Original Image
        plt.subplot(1, 2, 1)
        plt.imshow(img)
        plt.title(f"Original Image (True: {true_label})")
        plt.axis('off')
        
        # Image with Grad-CAM overlay
        plt.subplot(1, 2, 2)
        plt.imshow(superimposed_img)
        plt.title(f"Grad-CAM Overlay (True: {true_label})")
        plt.axis('off')
        
        plt.show()


